# Capacity--assignment expansion runs
Runs the multi-evidence controlled grid and fixed-spectrum causal interventions in pretrained and larger open-weight models. Use a GPU runtime and save every result to Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
assert os.path.isdir('/content/drive/MyDrive')

In [ ]:
import os, subprocess
repo = '/content/lengthgen'
if os.path.isdir(os.path.join(repo, '.git')):
    subprocess.run(['git', '-C', repo, 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/arkankau/lengthgen.git', repo], check=True)
os.chdir(repo)
subprocess.run(['pip', 'install', '-q', '-U', 'transformers>=5.0', 'accelerate', 'bitsandbytes', 'sentencepiece'], check=True)

## 1. Multi-evidence controlled grid

In [ ]:
multi_out = '/content/drive/MyDrive/lengthgen_expansion/multievidence_grid'
!python colab/paired_permutation_experiment.py --tasks pairadd --pes nope,rope --seeds 0,1,2,3 --lengths 10,25 --steps 4000 --warmup 400 --batch 256 --n-eval 256 --selection-examples 256 --bootstrap-draws 10000 --layers 2 --width 128 --heads 4 --mlp 512 --outdir {multi_out}
!python scripts/analyze_multievidence_routing.py {multi_out}/paired_permutation_results.json --require-complete --json-out {multi_out}/summary.json --report-out {multi_out}/summary.md

## 2. Validate the pretrained causal backend
The smoke must finish with invariant error near numerical zero before full runs.

In [ ]:
causal_root = '/content/drive/MyDrive/lengthgen_expansion/pretrained_causal'
!python colab/pretrained_causal_routing.py --model Qwen/Qwen2.5-1.5B --smoke --heads 4 --outdir {causal_root}/qwen1p5b_smoke

## 3. Causal family runs

In [ ]:
family = [
    ('pythia1p4b', 'EleutherAI/pythia-1.4b'),
    ('qwen1p5b', 'Qwen/Qwen2.5-1.5B'),
    ('gemma2b', 'google/gemma-2-2b'),
]
for tag, model in family:
    out = f'{causal_root}/{tag}'
    cmd = ['python', 'colab/pretrained_causal_routing.py', '--model', model, '--lengths', '5,20,80,160', '--n', '128', '--batch', '4', '--heads', '4', '--outdir', out]
    print('RUN', ' '.join(cmd))
    subprocess.run(cmd, check=True)

## 4. Larger-model causal runs
Start with Qwen-7B. Gemma and Llama may require accepting their licenses and logging into Hugging Face. Run one model at a time; use batch 1 on a T4.

In [ ]:
large = [
    ('qwen7b', 'Qwen/Qwen2.5-7B'),
    ('llama3b', 'meta-llama/Llama-3.2-3B'),
    ('gemma9b', 'google/gemma-2-9b'),
]
for tag, model in large:
    out = f'{causal_root}/{tag}'
    cmd = ['python', 'colab/pretrained_causal_routing.py', '--model', model, '--lengths', '5,20,80', '--n', '96', '--batch', '1', '--heads', '4', '--dtype', 'fp16', '--load-in-4bit', '--outdir', out]
    print('RUN', ' '.join(cmd))
    completed = subprocess.run(cmd)
    if completed.returncode:
        print('SKIP/FAILED', model, 'code', completed.returncode)

## 5. Collect summaries

In [ ]:
import glob
files = sorted(glob.glob(f'{causal_root}/*/pretrained_causal_routing_results.json'))
print('collected', len(files), 'runs')
for path in files: print(path)
if files:
    subprocess.run(['python', 'scripts/analyze_pretrained_causal_routing.py', *files, '--csv-out', f'{causal_root}/summary.csv', '--report-out', f'{causal_root}/summary.md'], check=True)
    print(open(f'{causal_root}/summary.md').read())